# 01 · BTP Orchestration Config 관리

**목표**: SAP AI Core Prompt Registry에 Orchestration config를 SDK로 수정(저장), 조회, 삭제하고 저장된 Config ID를 참조해 invoke합니다.

필수 환경 변수: `AICORE_BASE_URL`, `AICORE_AUTH_URL`, `AICORE_CLIENT_ID`, `AICORE_CLIENT_SECRET`, `AICORE_RESOURCE_GROUP`. `requirements.txt`의 `sap-ai-sdk-gen>=7.4.1,<8` 설치가 필요합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 저장된 Orchestration Config 관리 (UI)</h2>
<p style="margin: 7px 0;">Save는 Prompt Registry의 config를 생성하거나 같은 scenario/name/version을 수정합니다. Invoke는 저장된 Config ID를 사용합니다. Delete는 되돌릴 수 없으며 UI에서 <code>DELETE</code> 확인 입력이 필요합니다.</p>
</div>
</div>

In [2]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').is_dir())
sys.path[:0] = [str(project_root / 'shared'), str(project_root / '3_gen_ai_hub_services')]
from src.orchestration.registry_ui import build_orchestration_registry_ui

orchestration_registry_ui = build_orchestration_registry_ui()
orchestration_registry_ui.launch(inline=True, share=False, show_error=True, server_name='127.0.0.1', width='100%', height=900)

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ 체크포인트</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">저장 전과 후의 경계</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>구성 미리보기는 BTP에 쓰지 않습니다.</li><li>저장/수정과 삭제는 실제 Prompt Registry 리소스를 변경합니다.</li><li>Invoke는 저장된 ID의 spec을 BTP Orchestration deployment에서 실행합니다.</li></ul>
</div>
</div>

<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎 선택 심화</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">공개 core API로 관리하기</h2>
<p style="margin: 7px 0;">아래 셀은 UI가 호출하는 동일한 public API만 사용합니다. 저장된 config는 <code>config_id</code> reference로 invoke하며, 노트북에서 SDK 객체를 다시 조립하지 않습니다.</p>
</div>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').is_dir())
sys.path[:0] = [str(project_root / 'shared'), str(project_root / '3_gen_ai_hub_services')]
from lab_common.config import load_settings
from src.orchestration.lab_core import (delete_orchestration_config, invoke_saved_orchestration_config, preview_saved_orchestration_spec, save_orchestration_config)
settings = load_settings()

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 쓰기 전 spec 확인</h2>
<p style="margin: 7px 0;">먼저 JSON-friendly preview 결과로 prompt template과 model 설정을 확인합니다. 이 호출은 BTP 상태를 바꾸지 않습니다.</p>
</div>
</div>

In [ ]:
preview_saved_orchestration_spec(
    settings, system_prompt='You are a concise, helpful assistant.',
    user_prompt='{{?user_query}}', max_tokens=700,
)

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. config 저장 또는 수정</h2>
<p style="margin: 7px 0;">같은 scenario/name/version으로 다시 저장하면 Prompt Registry의 해당 config를 수정합니다. 아래 셀은 실제 BTP 쓰기 작업입니다.</p>
</div>
</div>

In [ ]:
saved = save_orchestration_config(
    settings, name='devday-orchestration-demo', version='1.0.0', scenario='devday',
    system_prompt='You are a concise, helpful assistant.', user_prompt='{{?user_query}}', max_tokens=700,
)
saved

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. 저장된 config invoke</h2>
<p style="margin: 7px 0;">core는 저장 응답의 <code>id</code>로 SDK <code>config_ref</code>를 만들고, RUNNING Orchestration deployment에 completion을 요청합니다.</p>
</div>
</div>

In [ ]:
invoke_saved_orchestration_config('Orchestration Service를 한 문장으로 설명해 주세요.', settings, config_id=saved['id'])

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. config 삭제</h2>
<p style="margin: 7px 0;">삭제는 되돌릴 수 없습니다. 기본값은 False라서 셀을 실행해도 삭제하지 않으며, 의도적으로 True로 바꾼 경우에만 public delete API를 호출합니다.</p>
</div>
</div>

In [ ]:
CONFIRM_DELETE = False  # 삭제 의도를 확인한 뒤에만 True로 변경합니다.
if CONFIRM_DELETE:
    delete_orchestration_config(saved['id'])
else:
    print('삭제하지 않았습니다. CONFIRM_DELETE=True로 바꾸면 saved config를 영구 삭제합니다.')